# Multi-Node Training Setup Guide

## Overview

Complete guide for setting up distributed training across multiple nodes.

### Topics Covered
- Environment setup
- Network configuration
- Launch scripts
- Troubleshooting

## 1. Environment Variables

```bash
# Required on each node
export MASTER_ADDR=<master_node_ip>
export MASTER_PORT=29500
export WORLD_SIZE=<total_gpus>
export RANK=<global_rank>  # Different per node
export LOCAL_RANK=<local_gpu_id>

# NCCL settings for multi-node
export NCCL_IB_DISABLE=0  # Enable InfiniBand
export NCCL_NET_GDR_LEVEL=2  # GPU Direct RDMA
export NCCL_DEBUG=INFO  # Debug logging
```

In [ ]:
import os
import torch.distributed as dist

def setup_multi_node():
    """Initialize distributed training for multi-node setup."""
    
    # Get environment variables
    master_addr = os.environ.get('MASTER_ADDR', 'localhost')
    master_port = os.environ.get('MASTER_PORT', '29500')
    world_size = int(os.environ.get('WORLD_SIZE', 1))
    rank = int(os.environ.get('RANK', 0))
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    
    # Initialize process group
    dist.init_process_group(
        backend='nccl',
        init_method=f'tcp://{master_addr}:{master_port}',
        world_size=world_size,
        rank=rank,
    )
    
    print(f"Rank {rank}/{world_size} initialized on {master_addr}")
    return rank, local_rank, world_size

## 2. Launch Scripts

### Using torchrun (Recommended)

In [ ]:
# Node 0 (Master)
launch_cmd_node0 = """
torchrun \\
    --nnodes=2 \\
    --nproc_per_node=8 \\
    --node_rank=0 \\
    --master_addr=192.168.1.100 \\
    --master_port=29500 \\
    train.py
"""

# Node 1
launch_cmd_node1 = """
torchrun \\
    --nnodes=2 \\
    --nproc_per_node=8 \\
    --node_rank=1 \\
    --master_addr=192.168.1.100 \\
    --master_port=29500 \\
    train.py
"""

print("Node 0 command:")
print(launch_cmd_node0)
print("\nNode 1 command:")
print(launch_cmd_node1)

## 3. Network Troubleshooting

| Issue | Symptom | Solution |
|-------|---------|----------|
| Firewall | Connection timeout | Open port 29500 |
| DNS | Cannot resolve hostname | Use IP addresses |
| NCCL | Hangs at init | Check NCCL_DEBUG output |
| Bandwidth | Slow training | Enable InfiniBand |

In [ ]:
def diagnose_network():
    """Diagnose common multi-node issues."""
    import socket
    
    checks = {
        'MASTER_ADDR': os.environ.get('MASTER_ADDR'),
        'MASTER_PORT': os.environ.get('MASTER_PORT'),
        'WORLD_SIZE': os.environ.get('WORLD_SIZE'),
        'RANK': os.environ.get('RANK'),
        'NCCL_DEBUG': os.environ.get('NCCL_DEBUG', 'Not set'),
    }
    
    print("Environment Check:")
    for k, v in checks.items():
        status = "OK" if v else "MISSING"
        print(f"  {k}: {v} [{status}]")
    
    # Test connectivity
    master = checks['MASTER_ADDR']
    port = int(checks['MASTER_PORT'] or 29500)
    
    if master:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(5)
            result = sock.connect_ex((master, port))
            sock.close()
            print(f"\nConnectivity to {master}:{port}: {'OK' if result == 0 else 'FAILED'}")
        except Exception as e:
            print(f"\nConnectivity test failed: {e}")

diagnose_network()

## 4. Summary

### Multi-Node Checklist

- [ ] All nodes can reach master IP:port
- [ ] Environment variables set correctly
- [ ] Same PyTorch/NCCL versions on all nodes
- [ ] InfiniBand enabled if available
- [ ] NCCL_DEBUG=INFO for troubleshooting